# Publish dbGaP Metadata Atlas from Colab

**Three cells: settings, local preview, explicit publish.** Upload the supplied code-only ZIP.
This does not mount or scan your Drive data folder, recollect data, or upload scientific reports.
Preview needs no GitHub token. Publishing requires a scoped token, a reviewed release/licensing
position, and a target-specific confirmation typed at runtime. See `docs/publishing.md` in the ZIP.

Use a personal or organizational GitHub owner you are authorized to publish under. The default
owner was resolved from the connected account during preparation. No repository was created
by preparing this notebook. After publication, the code and documentation will be public.


In [ ]:
# @title 1. Repository settings (preview by default)
GITHUB_OWNER = "conroyriodss"
REPOSITORY = "dbgap-metadata-atlas"
VISIBILITY = "public"  # Or private for a separate review-only target; never auto-changed.
BUNDLE_ZIP = ""  # Empty prompts for the supplied ZIP upload; no Drive mount required.
EXPECTED_BUNDLE_SHA256 = ""  # External bootstrap download includes its exact ZIP hash.
PUBLISH = False
PUBLIC_RELEASE_REVIEWED = False  # Includes authority to publish and the licensing position.
CREATE_IF_MISSING = True
OPEN_PULL_REQUEST = True  # Later changes to a populated repo use a review branch.
ENABLE_PAGES = False  # Optional documentation-only landing page.
SEED_QUESTIONS = False  # Optional ten unprioritized, deduplicated research issues.
NOTEBOOK_UPDATES = {}  # Optional exact local .ipynb paths, never an entire Drive folder.
# Example: {"notebooks/02_analyze.ipynb": "/content/my_reviewed_analysis.ipynb"}


In [ ]:
# @title 2. Upload, verify and preview locally (no GitHub calls)
from pathlib import Path, PurePosixPath
import hashlib, importlib, json, stat, sys, tempfile, zipfile

if not BUNDLE_ZIP:
    from google.colab import files as colab_files
    uploaded = colab_files.upload()
    choices = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(choices) != 1:
        raise ValueError("Upload exactly the supplied repository ZIP, not your data/archive folder.")
    BUNDLE_ZIP = str(Path(choices[0]).resolve())
    del uploaded
archive = Path(BUNDLE_ZIP)
actual_hash = hashlib.sha256(archive.read_bytes()).hexdigest()
if EXPECTED_BUNDLE_SHA256 and actual_hash != EXPECTED_BUNDLE_SHA256:
    raise ValueError("ZIP checksum mismatch. Do not run code from this archive.")
if not EXPECTED_BUNDLE_SHA256:
    print("No external ZIP hash configured. Use only a bundle you trust and have reviewed.")
print("ZIP SHA-256:", actual_hash)

# Inspect ZIP boundaries and every listed file hash before importing any bundled Python.
with zipfile.ZipFile(archive) as z:
    infos=z.infolist()
    if len(infos)>400 or sum(i.file_size for i in infos)>30*1024*1024:
        raise ValueError("Unexpectedly large code-only package.")
    seen=set()
    for i in infos:
        p=PurePosixPath(i.filename)
        if p.is_absolute() or ".." in p.parts or "\\" in i.filename or i.filename in seen:
            raise ValueError("Unsafe or duplicate ZIP member.")
        seen.add(i.filename)
        if stat.S_ISLNK(i.external_attr>>16):
            raise ValueError("ZIP contains a symbolic link.")
        if len(p.parts)<2 or p.parts[0]!="dbgap-metadata-atlas":
            raise ValueError("Wrong bundle structure.")
    prefix="dbgap-metadata-atlas/"
    manifest=json.loads(z.read(prefix+"provenance/package_manifest.json"))
    expected=set(prefix+n for n in manifest["files"])|{prefix+"provenance/package_manifest.json"}
    actual={i.filename for i in infos if not i.is_dir()}
    if expected!=actual:
        raise ValueError("ZIP files and the package manifest disagree.")
    for n,record in manifest["files"].items():
        content=z.read(prefix+n)
        if hashlib.sha256(content).hexdigest()!=record["sha256"] or len(content)!=record["bytes"]:
            raise ValueError("Package file checksum mismatch: "+n)
    stage=Path(tempfile.mkdtemp(prefix="dbgap-code-stage-"))
    z.extractall(stage)
REPO_DIR=stage/"dbgap-metadata-atlas"
sys.path.insert(0,str(REPO_DIR/"scripts"))
for name in ("repo_tools","publish_github"):
    sys.modules.pop(name,None)
from repo_tools import verify_package
from publish_github import preview, publish, build_payload, payload_fingerprint, confirmation_phrase
verify_package(REPO_DIR)
PREVIEW_REPORT, PREVIEW_PAYLOAD = preview(REPO_DIR,GITHUB_OWNER,REPOSITORY,VISIBILITY,NOTEBOOK_UPDATES)
PREVIEW_SIGNATURE=PREVIEW_REPORT["payload_sha256"]
print("Staged source/docs:", REPO_DIR)
print("Read LICENSE_STATUS.md and docs/publication_policy.md before publishing.")


In [ ]:
# @title 3. Publish only after review (disabled by default)
if not PUBLISH:
    print("Not published. Review the preview; set PUBLISH and PUBLIC_RELEASE_REVIEWED to True only when authorized.")
else:
    if not PUBLIC_RELEASE_REVIEWED:
        raise ValueError("Public-release and licensing review is required.")
    current=build_payload(REPO_DIR,GITHUB_OWNER,REPOSITORY,notebook_updates=NOTEBOOK_UPDATES)
    if payload_fingerprint(current)!=PREVIEW_SIGNATURE:
        raise ValueError("Files or target changed since preview. Run cell 2 again.")
    phrase=confirmation_phrase(GITHUB_OWNER,REPOSITORY,VISIBILITY)
    entered=input("To publish the reviewed source/docs, type exactly: "+phrase+"\n")
    result=publish(REPO_DIR,GITHUB_OWNER,REPOSITORY,visibility=VISIBILITY,
                   confirmation=entered,release_reviewed=PUBLIC_RELEASE_REVIEWED,
                   create_if_missing=CREATE_IF_MISSING,open_pull_request=OPEN_PULL_REQUEST,
                   enable_pages=ENABLE_PAGES,seed_questions=SEED_QUESTIONS,
                   notebook_updates=NOTEBOOK_UPDATES)
    print("Publication result:",json.dumps(result,indent=2))
    print("Open the actual GitHub URL, inspect Actions and any review PR, then revoke/narrow bootstrap token access.")
